<a href="https://colab.research.google.com/github/ericjenn/working-groups/blob/ericjenn-srpwg-wg1/safety-related-profile/documents/conv_specification_example/tests/conv_onnx.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install onnx onnxruntime

In [2]:
import math
import onnx
import onnxruntime as ort
import numpy as np
from onnx import TensorProto

# Define input and output tensor names
input1_name = "X"
maxpool_output_name1 = "Y"
maxpool_output_name2 = "Indices"


# Create the ONNX model with maxpool operator
def create_maxpool_model(input_shape, output_shape, dtype, dilation, paddings, k_shape, store_order=0):

    #Create "input-rank" input tensor
    input1 = onnx.helper.make_tensor_value_info(input1_name, dtype, input_shape)

    # Create output tensor (final result after maxpool operation)
    maxpool_output1 = onnx.helper.make_tensor_value_info(maxpool_output_name1, dtype, output_shape)
    maxpool_output2 = onnx.helper.make_tensor_value_info(maxpool_output_name2, TensorProto.INT64, output_shape)

    # Define MaxPool node
    maxpool_node = onnx.helper.make_node(
        "MaxPool",
        inputs=[input1_name],
        outputs=[maxpool_output_name1, maxpool_output_name2],
        ceil_mode = 0,
        dilations=dilation,
        kernel_shape=k_shape,
        pads=paddings,
        strides=[1, 1],
        auto_pad='NOTSET',
        storage_order=store_order
    )

    # Create the ONNX graph
    graph_def = onnx.helper.make_graph(
        [maxpool_node],
        "Test_MaxPool",
        [input1],
        [maxpool_output1, maxpool_output2],
    )

    # Create the ONNX model
    model = onnx.helper.make_model(graph_def, opset_imports=[onnx.helper.make_opsetid("", 22)]) # Explicitly set opset to 22
    model.ir_version = 10 
    onnx.checker.check_model(model)

    # Save the model
    onnx.save(model, "maxpool.onnx")

    # Load and run the model with ONNX Runtime
    session = ort.InferenceSession("maxpool.onnx")
    return session

def do_maxpool(x, session):
    # Run inference
    output = session.run(None, {input1_name: x})

    x_f = (np.array2string(x, separator=',', max_line_width=np.inf).replace('\n', '\n'))
    y_f = (np.array2string(output[0], separator=',', max_line_width=np.inf).replace('\n', '\n'))
    indices_f = (np.array2string(output[1], separator=',', max_line_width=np.inf).replace('\n', '\n'))

    # Display results
    print(f"X = \n{x_f}")
    print(f"Y = \n{y_f}")
    print(f"Indices = \n{indices_f}")



In [12]:
# Case noninal N1
onnx_type = onnx.TensorProto.DOUBLE
data = [[[[2.14485954, 3.60685315, 5.43428613],
          [0.24944269, 0.25525759, 1.95981565],
          [-1.57949968, 4.82017601, 3.77251085]]]]
x = np.array(data, dtype=np.float64)
y_shape=[1, 1, 2, 2]
dilation=[1,1]
pads = [0,0,0,0]
kernel_shape = [2,2]
session = create_maxpool_model(x.shape, y_shape, onnx_type, dilation, pads, kernel_shape)
do_maxpool(x, session)

X = 
[[[[ 2.14485954, 3.60685315, 5.43428613],
   [ 0.24944269, 0.25525759, 1.95981565],
   [-1.57949968, 4.82017601, 3.77251085]]]]
Y = 
[[[[3.60685315,5.43428613],
   [4.82017601,4.82017601]]]]
Indices = 
[[[[1,2],
   [7,7]]]]


In [6]:
# Case noninal N2: max in at least two positions in X
onnx_type = onnx.TensorProto.DOUBLE
data = [[[[3.60685315, 3.60685315, 5.43428613],
          [0.24944269, 0.25525759, 1.95981565],
          [-1.57949968, 4.82017601, 3.77251085]]]]
x = np.array(data, dtype=np.float64)
y_shape=[1, 1, 2, 2]
dilation=[1,1]
pads = [0,0,0,0]
kernel_shape = [2,2]
session = create_maxpool_model(x.shape, y_shape, onnx_type, dilation, pads, kernel_shape)
do_maxpool(x, session)

X = 
[[[[ 3.60685315, 3.60685315, 5.43428613],
   [ 0.24944269, 0.25525759, 1.95981565],
   [-1.57949968, 4.82017601, 3.77251085]]]]
Y = 
[[[[3.60685315,5.43428613],
   [4.82017601,4.82017601]]]]
Indices = 
[[[[0,2],
   [7,7]]]]


In [7]:
# Case noninal N3: max in at least two positions in X
onnx_type = onnx.TensorProto.DOUBLE
data = [[[[3.60685315, 3.60685315, 3.60685315],
          [0.24944269, 0.25525759, 1.95981565],
          [-1.57949968, 4.82017601, 3.77251085]]]]
x = np.array(data, dtype=np.float64)
y_shape=[1, 1, 2, 2]
dilation=[1,1]
pads = [0,0,0,0]
kernel_shape = [2,2]
session = create_maxpool_model(x.shape, y_shape, onnx_type, dilation, pads, kernel_shape)
do_maxpool(x, session)

X = 
[[[[ 3.60685315, 3.60685315, 3.60685315],
   [ 0.24944269, 0.25525759, 1.95981565],
   [-1.57949968, 4.82017601, 3.77251085]]]]
Y = 
[[[[3.60685315,3.60685315],
   [4.82017601,4.82017601]]]]
Indices = 
[[[[0,1],
   [7,7]]]]


In [8]:
# Case noninal N3: max in at least two positions in X
onnx_type = onnx.TensorProto.DOUBLE
data = [[[[3.60685315, 3.60685315, 3.60685315],
          [0.24944269, 0.25525759, 1.95981565],
          [-1.57949968, 4.82017601, 4.82017601]]]]
x = np.array(data, dtype=np.float64)
y_shape=[1, 1, 2, 2]
dilation=[1,1]
pads = [0,0,0,0]
kernel_shape = [2,2]
session = create_maxpool_model(x.shape, y_shape, onnx_type, dilation, pads, kernel_shape)
do_maxpool(x, session)

X = 
[[[[ 3.60685315, 3.60685315, 3.60685315],
   [ 0.24944269, 0.25525759, 1.95981565],
   [-1.57949968, 4.82017601, 4.82017601]]]]
Y = 
[[[[3.60685315,3.60685315],
   [4.82017601,4.82017601]]]]
Indices = 
[[[[0,1],
   [7,7]]]]


In [9]:
# Case noninal N4: max in at least two positions in X
onnx_type = onnx.TensorProto.DOUBLE
data = [[[[3.60685315, 3.60685315, 3.60685315],
          [3.60685315, 3.60685315, 3.60685315],
          [3.60685315, 3.60685315, 3.60685315]]]]
x = np.array(data, dtype=np.float64)
y_shape=[1, 1, 2, 2]
dilation=[1,1]
pads = [0,0,0,0]
kernel_shape = [2,2]
session = create_maxpool_model(x.shape, y_shape, onnx_type, dilation, pads, kernel_shape)
do_maxpool(x, session)

X = 
[[[[3.60685315,3.60685315,3.60685315],
   [3.60685315,3.60685315,3.60685315],
   [3.60685315,3.60685315,3.60685315]]]]
Y = 
[[[[3.60685315,3.60685315],
   [3.60685315,3.60685315]]]]
Indices = 
[[[[0,1],
   [3,4]]]]
